#### Custom Vision 으로 이미지 객체 인식 모델 만들기

1. 경기도 스쿨존 위도/경도 좌표를 이용해 카카오맵에서 로드뷰 이미지를 생성한다.
2. 해당 로드뷰 이미지에서 샘플 300~500개를 Custom Vision로 업로드한다.
3. road_area(차도영역), sidewalk_area(인도영역), parked_car(주정차차량) 태그를 생성하고 샘플이미지에 라벨링한다.
4. Quick Train 결과, Precision: 85.1% / Recall: 68.6% / mAP: 80.8% 으로 양호한 결과로 판단되므로 문서화한다.

In [ ]:
    import requests
    import os
    import pandas as pd

    PREDICTION_KEY = "key"  #customvision prediction key
    ENDPOINT = "https://eastus.api.cognitive.microsoft.com/customvision/v3.0/Prediction/1c8af540-ac3d-44a5-9fa4-66db9114b421/detect/iterations/SchoolzoneObjectDetection_Iteration1/image" #customvision prediction url

    headers = {
        "Prediction-Key": PREDICTION_KEY,
        "Content-Type": "application/octet-stream"
    }

    IMAGE_DIR = r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\roadview_images\object_detection_prediction"
    results = []

    for filename in os.listdir(IMAGE_DIR):
        if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
            continue

        path = os.path.join(IMAGE_DIR, filename)

        with open(path, "rb") as image_data:
            response = requests.post(
                ENDPOINT,
                headers=headers,
                data=image_data
            )

        if response.status_code != 200:
            print("Error:", response.status_code, response.text)
            continue

        data = response.json()
        predictions = data["predictions"]
        road_width = 0
        sidewalk_area = 0
        parked_count = 0

        for pred in predictions:
            if pred["probability"] < 0.5:
                continue

            tag = pred["tagName"]
            bbox = pred["boundingBox"]

            if tag == "road_area":
                road_width = bbox["width"]

            elif tag == "sidewalk_area":
                sidewalk_area += bbox["width"] * bbox["height"]

            elif tag == "parked_car":
                parked_count += 1

        results.append({
            "image": filename,
            "road_width_relative": road_width,
            "sidewalk_ratio": sidewalk_area,
            "parked_density": parked_count
        })

    df = pd.DataFrame(results)
    df.to_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\CustomVision\0_object_detection_features.csv", index=False)

    print("완료")

완료


In [ ]:
df.to_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\CustomVision\image_features.csv", index=False)


In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\CustomVision\0_object_detection_features.csv")

print(df.shape)
print(df.head())
print(df.isnull().sum())

(4015, 4)
                   image  road_width_relative  sidewalk_ratio  parked_density
0  AMI몬테소리손바닥어린이집_북쪽.png             0.400759        0.000000               1
1          AP어린이집_북쪽.png             0.428197        0.000000               2
2         EPS어린이집_북쪽.png             0.000000        0.000000               0
3     GC차일드케어어린이집_북쪽.png             0.402940        0.042165               0
4     LIG넥스원_어린이집_북쪽.png             0.345928        0.000000               0
image                  0
road_width_relative    0
sidewalk_ratio         0
parked_density         0
dtype: int64


In [2]:
import re

def label_from_name(name):
    base = name.replace(".png", "").replace(".jpg", "")
    
    # 숫자로 끝나는 경우 → 사고다발지
    if re.search(r"_\d+$", base):
        return 1
    else:
        return 0

df["accident_label"] = df["image"].apply(label_from_name)
df["accident_label"].value_counts()

accident_label
0    2284
1    1731
Name: count, dtype: int64

In [3]:
(df["road_width_relative"] == 0).sum()
df["is_zero"] = df["road_width_relative"] == 0
df.groupby("accident_label")["is_zero"].mean()

accident_label
0    0.167250
1    0.088388
Name: is_zero, dtype: float64

In [4]:
df = df[df["road_width_relative"] > 0]
df.groupby("accident_label")[[
    "road_width_relative",
    "sidewalk_ratio",
    "parked_density"
]].mean()

,road_width_relative,sidewalk_ratio,parked_density
accident_label,,,
0,0.379079,0.110589,0.446898
1,0.405309,0.104084,0.446768


In [ ]:
import requests
import pandas as pd

TRAINING_KEY = "key"
PROJECT_ID = "852e9554-f39a-4351-8289-b3b9096b555c"
ENDPOINT = f"https://eastus.api.cognitive.microsoft.com/customvision/v3.3/Training/projects/{PROJECT_ID}/images?take=1000"

headers = {"Training-Key": TRAINING_KEY}

tag_url = f"https://eastus.api.cognitive.microsoft.com/customvision/v3.3/Training/projects/{PROJECT_ID}/tags"

tag_response = requests.get(tag_url, headers=headers)
tags = tag_response.json()

# tagId → tagName 매핑
tag_map = {tag["id"]: tag["name"] for tag in tags}

print(tag_map)

{'4c91a91c-491f-4503-ad4a-1e3f1d487e5b': 'wide_lane', '25fe1a3b-d51c-4f94-b7a2-2935678754c7': 'barrier_no', '3cce918c-fe46-480b-9bc2-33c3e0b8bb9a': 'narrow_lane', '6bd1495a-33bb-4cb3-8a1b-3f3c2a04436f': 'barrier_yes'}


In [ ]:
import requests
import pandas as pd

TRAINING_KEY = "key"
PROJECT_ID = "852e9554-f39a-4351-8289-b3b9096b555c"

headers = {
    "Training-Key": TRAINING_KEY
}

# 1️⃣ 태그 매핑 가져오기
tag_url = f"https://eastus.api.cognitive.microsoft.com/customvision/v3.3/Training/projects/{PROJECT_ID}/tags"
tag_response = requests.get(tag_url, headers=headers)
tags = tag_response.json()

tag_map = {tag["id"]: tag["name"] for tag in tags}

records = []
skip = 0
batch_size = 256

# 2️⃣ tagged 이미지 가져오기
while True:
    url = f"https://eastus.api.cognitive.microsoft.com/customvision/v3.3/Training/projects/{PROJECT_ID}/images/tagged?take={batch_size}&skip={skip}"
    
    response = requests.get(url, headers=headers)
    data = response.json()
    
    if not data:
        break
    
    for img in data:
        image_name = img["originalImageUri"].split("/")[-1]
        
        lane_type = None
        barrier = None
        
        for tag in img["tags"]:
            tag_name = tag_map.get(tag["tagId"])
            
            if tag_name in ["single_lane", "two_lane", "multi_lane"]:
                lane_type = tag_name
            if tag_name in ["barrier_yes", "barrier_no"]:
                barrier = tag_name
        
        records.append({
            "image": image_name,
            "lane_type": lane_type,
            "barrier": barrier
        })
    
    skip += batch_size

labels_df = pd.DataFrame(records)
labels_df.to_csv("classification_labels.csv", index=False)

print("라벨 저장 완료:", len(labels_df))

라벨 저장 완료: 1247


In [34]:
features = pd.read_csv("image_features.csv")
labels = pd.read_csv("classification_labels.csv")

df = features.merge(labels, on="image", how="inner")

print(features["image"].head())
print(labels["image"].head())

0    AMI몬테소리손바닥어린이집_북쪽.png
1            AP어린이집_북쪽.png
2           EPS어린이집_북쪽.png
3       GC차일드케어어린이집_북쪽.png
4       LIG넥스원_어린이집_북쪽.png
Name: image, dtype: object
0    i-45abe09a420b47a3a46c2a72b28c2b23?skoid=ae506...
1    i-3346b656818d4fc5834dc4aba5831f15?skoid=ae506...
2    i-a4f15f9af6ca48d09ec31b45074bb577?skoid=ae506...
3    i-1e159d5e1cda48618c2db1f614802a63?skoid=ae506...
4    i-8f7a16c587d5447199f37449426f42f5?skoid=ae506...
Name: image, dtype: object


In [ ]:

df["lane_type"].value_counts()
df["barrier"].value_counts()

df.groupby("accident_label")["lane_type"].value_counts(normalize=True)
df.groupby("accident_label")["barrier"].value_counts(normalize=True)